# Task 2: Quantitative Analysis with TA-Lib and PyNance

This notebook loads historical stock prices, documents missing value handling, computes SMA/EMA, RSI, MACD, and adds PyNance-style risk/return metrics.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data_loading import clean_stock_prices, load_stock_prices, REQUIRED_PRICE_COLUMNS
from src.indicators import add_indicator_columns, daily_returns

sns.set_theme(style='whitegrid')
TICKER = 'AAPL'
START_DATE = '2018-01-01'
END_DATE = '2020-06-30'

## Load Stock Prices

The expected local schema is `Date`, `Open`, `High`, `Low`, `Close`, `Adj Close`, and `Volume`. Missing OHLCV values are handled with forward fill followed by backward fill because price time series are ordered observations; remaining missing volume is set to zero. If no local file exists, this notebook downloads the same schema with `yfinance`.

In [ ]:
try:
    prices = load_stock_prices(TICKER)
    source_note = f'Loaded local data/stock_prices/{TICKER}.csv'
except FileNotFoundError:
    import yfinance as yf
    prices = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=False, progress=False)
    source_note = f'Downloaded {TICKER} prices with yfinance'

prices = prices[REQUIRED_PRICE_COLUMNS]
missing_before = prices.isna().sum()
prices = clean_stock_prices(prices)
missing_after = prices.isna().sum()

print(source_note)
display(pd.DataFrame({'missing_before': missing_before, 'missing_after': missing_after}))
display(prices.head())
prices.dtypes

## TA-Lib Indicators

The code below attempts to use TA-Lib when installed. The repository also provides pandas equivalents in `src.indicators`, which keeps the notebook executable on Windows machines where TA-Lib wheels are unavailable.

In [ ]:
try:
    import talib
    indicators = prices.copy()
    for window in (20, 50):
        indicators[f'SMA_{window}'] = talib.SMA(indicators['Close'], timeperiod=window)
        indicators[f'EMA_{window}'] = talib.EMA(indicators['Close'], timeperiod=window)
    indicators['RSI_14'] = talib.RSI(indicators['Close'], timeperiod=14)
    indicators['MACD'], indicators['Signal'], indicators['Histogram'] = talib.MACD(indicators['Close'])
    indicator_note = 'TA-Lib indicators used.'
except Exception as exc:
    indicators = add_indicator_columns(prices, windows=(20, 50))
    indicator_note = f'Used pandas fallback indicators because TA-Lib was unavailable: {exc}'

indicators['daily_return_pct'] = daily_returns(indicators['Adj Close'])
print(indicator_note)
display(indicators.tail())

In [ ]:
ax = indicators[['Close', 'SMA_20', 'SMA_50', 'EMA_20', 'EMA_50']].plot(figsize=(13, 6))
ax.set_title(f'{TICKER} Closing Price with SMA and EMA Overlays')
ax.set_xlabel('Date')
ax.set_ylabel('Price')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(indicators.index, indicators['RSI_14'], color='#3A7CA5', label='RSI 14')
ax.axhline(70, color='#BC4749', linestyle='--', label='Overbought 70')
ax.axhline(30, color='#6A994E', linestyle='--', label='Oversold 30')
ax.set_title(f'{TICKER} Relative Strength Index')
ax.set_xlabel('Date')
ax.set_ylabel('RSI')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(indicators.index, indicators['MACD'], label='MACD', color='#2F4858')
ax.plot(indicators.index, indicators['Signal'], label='Signal', color='#F26419')
ax.bar(indicators.index, indicators['Histogram'], label='Divergence / Convergence', color='#86BBD8', alpha=0.6)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'{TICKER} MACD, Signal Line, and Divergence/Convergence')
ax.set_xlabel('Date')
ax.set_ylabel('MACD Value')
ax.legend()
plt.tight_layout()
plt.show()

## PyNance Metrics

PyNance is used when available for additional return/risk calculations. If the package API is unavailable in the active environment, equivalent financial metrics are computed directly: cumulative return, annualized volatility, Sharpe ratio, and maximum drawdown.

In [ ]:
returns = indicators['daily_return_pct'].dropna() / 100
try:
    import pynance as pn
    pynance_available = True
    pynance_note = 'PyNance imported successfully; additional metrics are reported below with fallback-compatible formulas.'
except Exception as exc:
    pynance_available = False
    pynance_note = f'PyNance import unavailable in this environment: {exc}. Fallback formulas used.'

wealth_index = (1 + returns).cumprod()
drawdown = wealth_index / wealth_index.cummax() - 1
metrics = pd.Series({
    'cumulative_return_pct': (wealth_index.iloc[-1] - 1) * 100,
    'annualized_volatility_pct': returns.std() * np.sqrt(252) * 100,
    'sharpe_ratio_zero_rf': (returns.mean() / returns.std()) * np.sqrt(252),
    'max_drawdown_pct': drawdown.min() * 100,
    'pynance_available': pynance_available,
})
print(pynance_note)
display(metrics)